# Mortgage Rate Enrichment

In [2]:
import pandas as pd

### Fetching the mortgage rate data from FRED

In [3]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [4]:
mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


### Resampling weekly rates to monthly averages

In [5]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')['rate_30yr_fixed']
.mean()
.reset_index()
)

In [6]:
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


### Creating a matching year_month key on the MLS datasets

In [7]:
sold = pd.read_csv("sold_residential.csv")
listings = pd.read_csv("listings_residential.csv")

C:\Users\mukun\AppData\Local\Temp\ipykernel_18872\3427280177.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("sold_residential.csv")
C:\Users\mukun\AppData\Local\Temp\ipykernel_18872\3427280177.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("listings_residential.csv")


In [8]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

In [9]:
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')

### Left Merge of Rates onto Datasets

In [10]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')

In [11]:
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

### Merge Validation

In [13]:
sold_with_rates['rate_30yr_fixed'].isnull().sum()

np.int64(0)

In [14]:
listings_with_rates['rate_30yr_fixed'].isnull().sum()

np.int64(0)

In [15]:
sold_with_rates[
['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2026-02-27,2026-02,1210000.0,6.0475
1,2026-02-24,2026-02,3150000.0,6.0475
2,2026-02-24,2026-02,2100000.0,6.0475
3,2026-02-27,2026-02,700000.0,6.0475
4,2026-02-27,2026-02,1191000.0,6.0475


### Exporting Enriched Datasets as CSV files 

In [16]:
sold_with_rates.to_csv("sold_with_rates.csv", index=False)

In [17]:
listings_with_rates.to_csv("listings_with_rates.csv", index=False)